# 观测站数据与 6 分钟雷达网格时空对齐

本 notebook 完成两件事：

1. 时间对齐：每个已插值到整 6 分钟的雷达文件，匹配同一小时的站点测风和降水文件。
2. 空间对齐：利用 `radar_situation.npz` 的 `lon/lat` 网格，把观测站点数据插值到与雷达一致的 `461 x 461` 空间网格。

输出为 `.npy` 文件，每个 6 分钟雷达时刻保存一个多通道数组：

- `channel 0`: 雷达反射率
- `channel 1`: 平均风速 `WIN_S_Avg_10mi` 插值网格
- `channel 2`: 平均风向对应的 `u` 分量
- `channel 3`: 平均风向对应的 `v` 分量
- `channel 4`: 瞬时最大风速 `WIN_S_Inst_Max` 插值网格
- `channel 5`: 逐小时降水 `PRE_1h` 插值网格

风向不直接做角度插值，而是先转成 `u/v` 风矢量后再插值，避免 359 度和 1 度这类角度跨界问题。

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

RAW_TEST_DIR = Path('test')
RADAR_6MIN_DIR = Path('test_radar_6min')
OUTPUT_DIR = Path('aligned_radar_station_npy')

OUTPUT_DIR.mkdir(exist_ok=True)

assert RAW_TEST_DIR.exists(), f'目录不存在: {RAW_TEST_DIR.resolve()}'
assert RADAR_6MIN_DIR.exists(), f'目录不存在: {RADAR_6MIN_DIR.resolve()}，请先运行雷达 6 分钟插值 notebook'

# IDW 插值参数。
K_NEIGHBORS = 8
IDW_POWER = 2.0

RAW_TEST_DIR.resolve(), RADAR_6MIN_DIR.resolve(), OUTPUT_DIR.resolve()

## 1. 读取雷达经纬度网格

In [ ]:
radar_geo = np.load(RAW_TEST_DIR / 'radar_situation.npz', allow_pickle=False)
radar_lon = radar_geo['lon'].astype(np.float64)
radar_lat = radar_geo['lat'].astype(np.float64)

grid_shape = radar_lon.shape
target_points = np.column_stack([radar_lon.ravel(), radar_lat.ravel()])

lon_min, lon_max = float(radar_lon.min()), float(radar_lon.max())
lat_min, lat_max = float(radar_lat.min()), float(radar_lat.max())

print('radar grid shape:', grid_shape)
print('lon range:', lon_min, lon_max)
print('lat range:', lat_min, lat_max)

## 2. 建立时间索引

雷达文件已经是整 6 分钟。站点文件是整小时，因此每个雷达时刻匹配其所在小时的站点文件。

In [ ]:
def parse_radar_6min_time(path: Path) -> datetime:
    timestamp = path.stem.split('_', 1)[1]
    return datetime.strptime(timestamp, '%Y%m%d%H%M%S')


def parse_hourly_time(path: Path, prefix: str) -> datetime:
    timestamp = path.stem[len(prefix):]
    return datetime.strptime(timestamp, '%Y%m%d%H')


radar_files = sorted(RADAR_6MIN_DIR.glob('Z9002_*.npy'), key=parse_radar_6min_time)
radar_times = [parse_radar_6min_time(path) for path in radar_files]

wind_files_by_hour = {
    parse_hourly_time(path, 'wind_'): path
    for path in RAW_TEST_DIR.glob('wind_*.parquet')
}
pre_files_by_hour = {
    parse_hourly_time(path, 'pre_1h_'): path
    for path in RAW_TEST_DIR.glob('pre_1h_*.parquet')
}

time_index = pd.DataFrame(
    {
        'radar_time': radar_times,
        'radar_file': [path.name for path in radar_files],
    }
)
time_index['obs_hour'] = time_index['radar_time'].dt.floor('h')
time_index['has_wind'] = time_index['obs_hour'].isin(wind_files_by_hour)
time_index['has_pre'] = time_index['obs_hour'].isin(pre_files_by_hour)
time_index['ready'] = time_index['has_wind'] & time_index['has_pre']

print('6 分钟雷达帧数:', len(time_index))
print('可匹配站点小时数据的雷达帧数:', int(time_index['ready'].sum()))
time_index.head(), time_index.tail()

## 3. 清洗站点数据并做 IDW 空间插值

In [ ]:
def clean_station_table(df: pd.DataFrame, value_columns):
    """保留雷达范围附近的站点，并把坐标和值统一转成数值。"""
    cols = ['Lon', 'Lat'] + value_columns
    out = df[cols].copy()
    for col in cols:
        out[col] = pd.to_numeric(out[col], errors='coerce')

    out = out.dropna(subset=['Lon', 'Lat'])

    # 先按雷达外包框筛选。IDW 只使用雷达范围内站点，避免远处站点影响结果。
    out = out[
        out['Lon'].between(lon_min, lon_max)
        & out['Lat'].between(lat_min, lat_max)
    ].copy()

    return out


def idw_interpolate_to_grid(station_df: pd.DataFrame, value_col: str):
    """用经纬度上的 K 近邻 IDW 将站点值插值到雷达网格。"""
    data = station_df[['Lon', 'Lat', value_col]].dropna().copy()
    if data.empty:
        return np.full(grid_shape, np.nan, dtype=np.float32)

    source_points = data[['Lon', 'Lat']].to_numpy(dtype=np.float64)
    source_values = data[value_col].to_numpy(dtype=np.float32)

    k = min(K_NEIGHBORS, len(data))
    tree = cKDTree(source_points)
    distances, indices = tree.query(target_points, k=k)

    if k == 1:
        distances = distances[:, None]
        indices = indices[:, None]

    exact = distances == 0
    weights = 1.0 / np.maximum(distances, 1e-12) ** IDW_POWER
    neighbor_values = source_values[indices]

    weighted = np.sum(weights * neighbor_values, axis=1) / np.sum(weights, axis=1)

    # 如果目标格点刚好有站点，直接使用该站点值。
    exact_rows = exact.any(axis=1)
    if exact_rows.any():
        first_exact = exact[exact_rows].argmax(axis=1)
        weighted[exact_rows] = neighbor_values[exact_rows, first_exact]

    return weighted.reshape(grid_shape).astype(np.float32)


def wind_dir_speed_to_uv(speed, direction_degree):
    """气象风向转 u/v。风向表示风从哪个方向吹来。"""
    radians = np.deg2rad(direction_degree)
    u = -speed * np.sin(radians)
    v = -speed * np.cos(radians)
    return u, v

## 4. 对单个小时生成站点网格

同一小时内会对应多个 6 分钟雷达时刻，因此站点网格按小时缓存，避免重复插值。

In [ ]:
station_grid_cache = {}


def make_station_grids_for_hour(obs_hour: pd.Timestamp):
    obs_hour = obs_hour.to_pydatetime() if hasattr(obs_hour, 'to_pydatetime') else obs_hour
    if obs_hour in station_grid_cache:
        return station_grid_cache[obs_hour]

    wind_path = wind_files_by_hour[obs_hour]
    pre_path = pre_files_by_hour[obs_hour]

    wind_df = pd.read_parquet(wind_path)
    pre_df = pd.read_parquet(pre_path)

    wind_df = clean_station_table(
        wind_df,
        ['WIN_S_Avg_10mi', 'WIN_D_Avg_10mi', 'WIN_S_Inst_Max'],
    )
    pre_df = clean_station_table(pre_df, ['PRE_1h'])

    wind_df['wind_u_avg_10mi'], wind_df['wind_v_avg_10mi'] = wind_dir_speed_to_uv(
        wind_df['WIN_S_Avg_10mi'],
        wind_df['WIN_D_Avg_10mi'],
    )

    grids = {
        'wind_speed_avg_10mi': idw_interpolate_to_grid(wind_df, 'WIN_S_Avg_10mi'),
        'wind_u_avg_10mi': idw_interpolate_to_grid(wind_df, 'wind_u_avg_10mi'),
        'wind_v_avg_10mi': idw_interpolate_to_grid(wind_df, 'wind_v_avg_10mi'),
        'wind_speed_inst_max': idw_interpolate_to_grid(wind_df, 'WIN_S_Inst_Max'),
        'pre_1h': idw_interpolate_to_grid(pre_df, 'PRE_1h'),
        'wind_station_count': len(wind_df),
        'pre_station_count': len(pre_df),
    }

    station_grid_cache[obs_hour] = grids
    return grids


# 测试第一个可匹配小时
first_ready_hour = time_index.loc[time_index['ready'], 'obs_hour'].iloc[0]
test_grids = make_station_grids_for_hour(first_ready_hour)
print('测试小时:', first_ready_hour)
print('wind station count:', test_grids['wind_station_count'])
print('pre station count:', test_grids['pre_station_count'])
print('wind grid shape:', test_grids['wind_speed_avg_10mi'].shape)
print('pre grid shape:', test_grids['pre_1h'].shape)

## 5. 批量输出对齐后的 `.npy`

每个输出文件对应一个 6 分钟雷达时刻，数组形状为：`(6, 461, 461)`。

In [ ]:
CHANNELS = [
    'radar_reflectivity',
    'wind_speed_avg_10mi',
    'wind_u_avg_10mi',
    'wind_v_avg_10mi',
    'wind_speed_inst_max',
    'pre_1h',
]


def output_name_for_time(t: datetime) -> str:
    return f'aligned_{t:%Y%m%d%H%M%S}.npy'


records = []
ready_rows = time_index[time_index['ready']].copy()

for n, row in enumerate(ready_rows.itertuples(index=False), start=1):
    radar_time = row.radar_time.to_pydatetime() if hasattr(row.radar_time, 'to_pydatetime') else row.radar_time
    obs_hour = row.obs_hour.to_pydatetime() if hasattr(row.obs_hour, 'to_pydatetime') else row.obs_hour
    radar_path = RADAR_6MIN_DIR / row.radar_file
    out_path = OUTPUT_DIR / output_name_for_time(radar_time)

    radar_arr = np.load(radar_path, allow_pickle=False).astype(np.float32, copy=False)
    station_grids = make_station_grids_for_hour(obs_hour)

    aligned = np.stack(
        [
            radar_arr,
            station_grids['wind_speed_avg_10mi'],
            station_grids['wind_u_avg_10mi'],
            station_grids['wind_v_avg_10mi'],
            station_grids['wind_speed_inst_max'],
            station_grids['pre_1h'],
        ],
        axis=0,
    ).astype(np.float32)

    np.save(out_path, aligned)

    records.append(
        {
            'radar_time': radar_time,
            'obs_hour': obs_hour,
            'output_file': out_path.name,
            'radar_file': row.radar_file,
            'wind_file': wind_files_by_hour[obs_hour].name,
            'pre_file': pre_files_by_hour[obs_hour].name,
            'shape': str(aligned.shape),
            'wind_station_count': station_grids['wind_station_count'],
            'pre_station_count': station_grids['pre_station_count'],
        }
    )

    if n % 50 == 0 or n == len(ready_rows):
        print(f'{n}/{len(ready_rows)} saved')

alignment_log = pd.DataFrame(records)
alignment_log.to_csv(OUTPUT_DIR / 'alignment_log.csv', index=False)
pd.Series(CHANNELS, name='channel_name').to_csv(OUTPUT_DIR / 'channels.csv', index_label='channel')

alignment_log.head(), alignment_log.tail()

## 6. 检查输出

In [ ]:
saved_files = sorted(OUTPUT_DIR.glob('aligned_*.npy'))
print('输出 aligned npy 数量:', len(saved_files))

sample = np.load(saved_files[0], allow_pickle=False)
print('样例文件:', saved_files[0].name)
print('shape:', sample.shape)
print('dtype:', sample.dtype)

for idx, name in enumerate(CHANNELS):
    arr = sample[idx]
    print(idx, name, 'nan:', int(np.isnan(arr).sum()), 'min:', float(np.nanmin(arr)), 'max:', float(np.nanmax(arr)))

## 7. 读取对齐结果示例

In [ ]:
aligned = np.load(saved_files[0], allow_pickle=False)

radar_reflectivity = aligned[0]
wind_speed_avg_10mi = aligned[1]
wind_u_avg_10mi = aligned[2]
wind_v_avg_10mi = aligned[3]
wind_speed_inst_max = aligned[4]
pre_1h = aligned[5]

radar_reflectivity.shape, wind_speed_avg_10mi.shape, pre_1h.shape